
# 🔥 Advanced AML Risk Scoring & Analyst Optimization System (Presentation Version)

This notebook includes:

- Synthetic AML alert dataset
- Feature engineering
- ML model training
- Accuracy score
- Confusion matrix visualization
- ROC curve
- AUC score
- Feature importance visualization
- Analyst workload distribution visualization

This version is suitable for demonstrating to leadership.


In [ ]:
!pip install rapidfuzz scikit-learn pandas numpy matplotlib joblib

In [ ]:

import pandas as pd
import numpy as np
from rapidfuzz import fuzz
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_curve, auc
import matplotlib.pyplot as plt
import joblib
np.random.seed(42)


## Feature Engineering

In [ ]:

def normalize(text):
    return text.lower().strip()

def name_similarity(a, b):
    return fuzz.token_sort_ratio(a, b) / 100.0

def generate_features(name_sim, dob, country, gender, severity):
    return [name_sim, dob, country, gender, severity]


## Generate Synthetic AML Dataset

In [ ]:

data = []
labels = []

for _ in range(1500):
    name_sim = np.random.uniform(0.4,1.0)
    dob = np.random.choice([0,1], p=[0.7,0.3])
    country = np.random.choice([0,1], p=[0.6,0.4])
    gender = np.random.choice([0,1], p=[0.5,0.5])
    severity = np.random.uniform(0.3,1.0)

    label = 1 if (name_sim>0.85 and dob==1 and country==1) else 0

    data.append([name_sim,dob,country,gender,severity])
    labels.append(label)

X = np.array(data)
y = np.array(labels)

print("Dataset Size:", X.shape)


## Train Logistic Regression Model

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2)

model = LogisticRegression()
model.fit(X_train,y_train)

preds = model.predict(X_test)
probs = model.predict_proba(X_test)[:,1]

accuracy = accuracy_score(y_test,preds)
print("Accuracy Score:", accuracy)


## Confusion Matrix Visualization

In [ ]:

cm = confusion_matrix(y_test,preds)

plt.figure()
plt.imshow(cm)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.colorbar()
plt.show()


## ROC Curve & AUC Score

In [ ]:

fpr, tpr, thresholds = roc_curve(y_test,probs)
roc_auc = auc(fpr,tpr)

plt.figure()
plt.plot(fpr,tpr)
plt.title("ROC Curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.show()

print("AUC Score:", roc_auc)


## Feature Importance (Explainability)

In [ ]:

importance = model.coef_[0]
feature_names = ["NameSim","DOB","Country","Gender","Severity"]

plt.figure()
plt.barh(feature_names,importance)
plt.title("Feature Importance")
plt.show()


## Analyst Workload Distribution Simulation

In [ ]:

def decision_logic(prob):
    if prob > 0.8:
        return "HIGH"
    elif prob > 0.4:
        return "MEDIUM"
    else:
        return "LOW"

workload = [decision_logic(p) for p in probs]

counts = pd.Series(workload).value_counts()

plt.figure()
counts.plot(kind='bar')
plt.title("Alert Priority Distribution")
plt.xlabel("Priority Level")
plt.ylabel("Number of Alerts")
plt.show()

counts
